In [83]:
!mkdir ../data
!unzip -o ../FinalFinancial.csv.zip -d ../data

mkdir: cannot create directory ‘../data’: File exists
Archive:  ../FinalFinancial.csv.zip
  inflating: ../data/FinalFinancial.csv  


# Dependencies

In [84]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

from catboost import CatBoostClassifier

SEED = 6967

In [85]:
df = pd.read_csv('../data/FinalFinancial.csv')
df.head()

,Title,Content,text,length,is_true
0,"TSX Slightly Down, Books Weekly Gains","TSX Slightly Down, Books Weekly GainsUnited St...","Title:TSX Slightly Down, Books Weekly Gains\nC...",1020,1
1,UnitedHealth Hits 4-week High,UnitedHealth Hits 4-week HighUnited States sto...,Title:UnitedHealth Hits 4-week High\nContent:U...,152,1
2,Cisco Systems Hits 4-week Low,Cisco Systems Hits 4-week LowUnited States sto...,Title:Cisco Systems Hits 4-week Low\nContent:C...,151,1
3,AT&T Hits All-time Low,AT&T Hits All-time LowUnited States stocksAT&T...,Title:AT&T Hits All-time Low\nContent:AT&T Hit...,131,1
4,Microsoft Hits 4-week High,Microsoft Hits 4-week HighUnited States stocks...,Title:Microsoft Hits 4-week High\nContent:Micr...,143,1


In [86]:
df.isna().sum()

Title      0
Content    0
text       0
length     0
is_true    0
dtype: int64

# Data split

In [87]:
df_train_val, df_test = train_test_split(df, test_size=0.3, random_state=SEED, shuffle=True, stratify=df['is_true'])
df_train, df_val = train_test_split(df_train_val, test_size=0.3, random_state=SEED, shuffle=True, stratify=df_train_val['is_true'])

X_train, y_train = df_train[['text']], df_train['is_true']
X_val, y_val = df_val[['text']], df_val['is_true']
X_test, y_test = df_test[['text']], df_test['is_true']

df_train.shape, df_val.shape, df_test.shape

((97, 5), (42, 5), (60, 5))

# Models

## Catboost Classifier based on text features

In [88]:
model = CatBoostClassifier(
    iterations=1000,
    verbose=10,
    eval_metric='AUC',
    early_stopping_rounds=100,
    text_features=['text'],
    random_seed=SEED,
    use_best_model=True
)

model.fit(X_train, y_train, eval_set=(X_val, y_val))

Learning rate set to 0.017809
0:	test: 0.9047619	best: 0.9047619 (0)	total: 8.76ms	remaining: 8.75s
10:	test: 0.9727891	best: 0.9954649 (3)	total: 121ms	remaining: 10.9s
20:	test: 0.9886621	best: 0.9954649 (3)	total: 191ms	remaining: 8.9s
30:	test: 0.9977324	best: 0.9977324 (23)	total: 258ms	remaining: 8.05s
40:	test: 0.9977324	best: 0.9977324 (23)	total: 329ms	remaining: 7.69s
50:	test: 1.0000000	best: 1.0000000 (42)	total: 398ms	remaining: 7.4s
60:	test: 1.0000000	best: 1.0000000 (42)	total: 466ms	remaining: 7.18s
70:	test: 1.0000000	best: 1.0000000 (42)	total: 536ms	remaining: 7.01s
80:	test: 1.0000000	best: 1.0000000 (42)	total: 603ms	remaining: 6.84s
90:	test: 1.0000000	best: 1.0000000 (42)	total: 670ms	remaining: 6.69s
100:	test: 1.0000000	best: 1.0000000 (42)	total: 737ms	remaining: 6.56s
110:	test: 1.0000000	best: 1.0000000 (42)	total: 803ms	remaining: 6.43s
120:	test: 0.9977324	best: 1.0000000 (42)	total: 870ms	remaining: 6.32s
130:	test: 0.9977324	best: 1.0000000 (42)	total: 

CatBoostClassifier(early_stopping_rounds=100, eval_metric='AUC', iterations=1000, random_seed=6967, text_features=['text'], use_best_model=True, verbose=10)

In [89]:
test_pred = model.predict_proba(X_test)[:, 1]
test_pred = np.clip(test_pred, 0, 1)

print(classification_report(y_test, model.predict(X_test)))
print(roc_auc_score(y_test, test_pred))

              precision    recall  f1-score   support

           0       1.00      0.90      0.95        30
           1       0.91      1.00      0.95        30

    accuracy                           0.95        60
   macro avg       0.95      0.95      0.95        60
weighted avg       0.95      0.95      0.95        60

0.9922222222222222
